## SelfGNN — Yelp Merchant Dataset Preprocessing

**Input:** `datasetRaw/yelp/` (Yelp Academic Dataset)
- `yelp_academic_dataset_business.json` — business metadata
- `yelp_academic_dataset_review.json` — user reviews with timestamps and stars

**Output:** `Datasets/yelp-merchant/`

| File | Description |
|------|-------------|
| `trn_mat_time` | `[global_mat, subMat (T=5), timeMat]` pickle |
| `sequence` | Per-user chronological merchant visit list |
| `tst_int` / `val_int` | Held-out positive merchant per user |
| `test_dict` / `val_dict` | `{uid → [neg_merchants]}` (999 each) |
| `user2id.pkl` / `merchant2id.pkl` | ID mappings for feature extraction |
| `train/test/val_yelp_merchant.csv` | TSV exports |
| `edge_weights.pkl` | `{(uid,mid) → avg_stars}` for `--use_edge_features` |

**Configuration:** `MIN_INTERACTIONS=5` (5-core), `GRAPH_NUM=5` time slices, `TEST_NEG=999`

**Prerequisite:** Run this notebook before `feature_extraction_yelp.ipynb`

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR           = '../Datasets/yelp-merchant/'
RAW_DIR           = '../datasetRaw/yelp/'
MIN_INTERACTIONS  = 5
GRAPH_NUM         = 8
TEST_NEG          = 999  # negative merchants per eval user
PICK_NUM          = 10_000  # max users sampled for test/val
SEED              = 100

# ── Imports ────────────────────────────────────────────────────────────────────
import os, json, pickle, copy, ast
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import scipy.sparse as sp

os.makedirs(OUT_DIR, exist_ok=True)
print('Libraries loaded.')
print('Output dir :', OUT_DIR)
print('Raw dir    :', RAW_DIR)

Libraries loaded.
Output dir : ../Datasets/yelp-merchant/
Raw dir    : ../datasetRaw/yelp/


In [10]:
# ── Cell 3: Load businesses ─────────────────────────────────────────────────────
# Keep every business that has at least one non-empty category token.
# This excludes purely service entries with no category annotation.

merchant_bids = set()
bid_to_name   = {}
cat_tokens = set()

business_path = RAW_DIR + 'yelp_academic_dataset_business.json'
with open(business_path, 'r', encoding='utf-8') as f:
    for line in f:
        b = json.loads(line)
        cats = b.get('categories') or ''
        cat_tokens.update({c.strip() for c in cats.split(',') if c.strip()})
        if cat_tokens:
            bid = b['business_id']
            merchant_bids.add(bid)
            bid_to_name[bid] = b.get('name', '')

print(f'Merchant businesses found : {len(merchant_bids):,}')
print('Example names:', list(bid_to_name.values())[:5])
print('Example category tokens:', len(cat_tokens), list(cat_tokens)[:1400])

Merchant businesses found : 150,346
Example names: ['Abby Rappoport, LAC, CMQ', 'The UPS Store', 'Target', 'St Honore Pastries', 'Perkiomen Valley Brewery']
Example category tokens: 1311 ['Musical Instrument Services', 'Candle Stores', 'Auto Parts & Supplies', 'Neuropathologists', 'Martial Arts', 'General Festivals', 'Powder Coating', 'Trinidadian', 'Boating', 'Cheerleading', 'Knitting Supplies', 'Psychic Mediums', 'Internet Service Providers', 'Body Shops', 'Mass Media', 'Honey', 'Bike tours', 'Ophthalmologists', 'Ziplining', 'Books', 'Grilling Equipment', 'Sleep Specialists', 'Immunodermatologists', 'Hair Removal', 'Commercial Truck Repair', 'Concierge Medicine', 'Outdoor Power Equipment Services', 'Thai', 'Vehicle Wraps', 'Backflow Services', 'Auto Insurance', 'Haitian', 'Israeli', 'Party Equipment Rentals', 'Ticket Sales', 'Diamond Buyers', 'Cheese Shops', 'Piano Services', 'Dart Arenas', 'Montessori Schools', 'Opera & Ballet', 'Water Heater Installation/Repair', 'Donairs', 'Allerg

In [3]:
# ── Cell 4: Load reviews ────────────────────────────────────────────────────────
# Streams the review JSON line-by-line to avoid loading 6M records into RAM at once.
# Stores: raw_interactions[user_id][business_id] = [unix_timestamp, ...]
# Also stores star ratings per (user, merchant) for edge_weights output.

def parse_date(date_str: str) -> int | None:
    """Parse Yelp date 'YYYY-MM-DD HH:MM:SS' → Unix timestamp."""
    try:
        return int(datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S').timestamp())
    except Exception:
        return None

raw_interactions: dict[str, dict[str, list[int]]] = defaultdict(lambda: defaultdict(list))
raw_stars: dict[str, dict[str, list[float]]]      = defaultdict(lambda: defaultdict(list))

n_read = n_kept = 0

review_path = RAW_DIR + 'yelp_academic_dataset_review.json'
with open(review_path, 'r', encoding='utf-8') as f:
    for line in f:
        n_read += 1
        rv  = json.loads(line)
        bid = rv.get('business_id', '')
        if bid not in merchant_bids:
            continue
        uid = rv.get('user_id', '')
        ts  = parse_date(rv.get('date', ''))
        if uid and bid and ts:
            raw_interactions[uid][bid].append(ts)
            raw_stars[uid][bid].append(float(rv.get('stars', 0.0)))
            n_kept += 1
        if n_read % 1_000_000 == 0:
            print(f'  Read {n_read:,} reviews, kept {n_kept:,}')

print(f'\nTotal reviews read : {n_read:,}')
print(f'Merchant reviews   : {n_kept:,}')
print(f'Unique users       : {len(raw_interactions):,}')
print(f'Unique merchants   : {len({b for u in raw_interactions.values() for b in u}):,}')

  Read 1,000,000 reviews, kept 999,934
  Read 2,000,000 reviews, kept 1,999,808
  Read 3,000,000 reviews, kept 2,999,710
  Read 4,000,000 reviews, kept 3,999,622
  Read 5,000,000 reviews, kept 4,999,524
  Read 6,000,000 reviews, kept 5,999,455

Total reviews read : 6,990,280
Merchant reviews   : 6,989,591
Unique users       : 1,987,685
Unique merchants   : 150,243


In [4]:
# ── Cell 5: K-core filter + ID mapping ─────────────────────────────────────────
# Iteratively removes users/merchants with fewer than MIN_INTERACTIONS interactions
# until convergence (5-core). Then assigns contiguous integer IDs.

def kcore_filter(
    interactions: dict,
    min_u: int = MIN_INTERACTIONS,
    min_i: int = MIN_INTERACTIONS,
) -> dict:
    """Return filtered interactions dict satisfying k-core constraint."""
    data = {u: dict(merchants) for u, merchants in interactions.items()}
    for iteration in range(1000):
        merchant_cnt: dict[str, int] = defaultdict(int)
        for merchants in data.values():
            for bid in merchants:
                merchant_cnt[bid] += 1
        valid_merchants = {bid for bid, cnt in merchant_cnt.items() if cnt >= min_i}

        data = {
            u: {bid: ts for bid, ts in merchants.items() if bid in valid_merchants}
            for u, merchants in data.items()
        }
        data = {u: merchants for u, merchants in data.items() if len(merchants) >= min_u}

        merchant_cnt2: dict[str, int] = defaultdict(int)
        for merchants in data.values():
            for bid in merchants:
                merchant_cnt2[bid] += 1
        still_valid = {bid for bid, cnt in merchant_cnt2.items() if cnt >= min_i}

        n_u = len(data)
        n_i = len(still_valid)
        n_e = sum(len(v) for v in data.values())
        print(f'  Iter {iteration+1}: {n_u:,} users, {n_i:,} merchants, {n_e:,} pairs')

        if still_valid == valid_merchants and all(len(m) >= min_u for m in data.values()):
            break
        if n_u == 0 or n_i == 0:
            break
    return data

print(f'K-core filtering (min={MIN_INTERACTIONS})...')
filtered = kcore_filter(raw_interactions)

all_merchants_kcore = {bid for u in filtered.values() for bid in u}
print(f'After k-core: {len(filtered):,} users, {len(all_merchants_kcore):,} merchants')

# ── Assign IDs ──────────────────────────────────────────────────────────────────
user_strs     = sorted(filtered.keys())
merchant_strs = sorted(all_merchants_kcore)

user2id     = {u: i for i, u in enumerate(user_strs)}
merchant2id = {m: i for i, m in enumerate(merchant_strs)}

usrnum      = len(user_strs)
merchantnum = len(merchant_strs)

with open(OUT_DIR + 'user2id.pkl', 'wb') as f:
    pickle.dump(user2id, f)
with open(OUT_DIR + 'merchant2id.pkl', 'wb') as f:
    pickle.dump(merchant2id, f)
print(f'Saved: user2id.pkl ({usrnum:,}), merchant2id.pkl ({merchantnum:,})')

# ── Build integer-keyed interaction dict ────────────────────────────────────────
interaction: list[dict[int, list[int]] | None] = [None] * usrnum
minn = int(2e12)
maxx = 0

for u_str, merchants in filtered.items():
    u = user2id[u_str]
    interaction[u] = {}
    for bid, tss in merchants.items():
        if bid not in merchant2id:
            continue
        mid = merchant2id[bid]
        interaction[u][mid] = sorted(tss)
        for ts in tss:
            if ts < minn: minn = ts
            if ts > maxx: maxx = ts

n_pairs  = sum(len(v) for v in interaction if v is not None)
n_events = sum(sum(len(tss) for tss in v.values()) for v in interaction if v is not None)
density  = n_pairs / (usrnum * merchantnum) * 100

print(f'usrnum      = {usrnum:,}')
print(f'merchantnum = {merchantnum:,}')
print(f'Unique pairs: {n_pairs:,} | Total events: {n_events:,}')
print(f'Density     : {density:.4f}%')
print(f'Time range  : {datetime.fromtimestamp(minn)} → {datetime.fromtimestamp(maxx)}')

K-core filtering (min=5)...
  Iter 1: 279,089 users, 110,587 merchants, 4,153,042 pairs
  Iter 2: 269,148 users, 109,374 merchants, 4,006,537 pairs
  Iter 3: 268,668 users, 109,327 merchants, 3,999,831 pairs
  Iter 4: 268,653 users, 109,325 merchants, 3,999,583 pairs
  Iter 5: 268,653 users, 109,325 merchants, 3,999,575 pairs
After k-core: 268,653 users, 109,325 merchants
Saved: user2id.pkl (268,653), merchant2id.pkl (109,325)
usrnum      = 268,653
merchantnum = 109,325
Unique pairs: 3,999,575 | Total events: 4,190,201
Density     : 0.0136%
Time range  : 2005-02-16 03:23:22 → 2022-01-19 19:48:45


In [5]:
# ── Cell 6: Dataset statistics ──────────────────────────────────────────────────

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    user_degrees     = [len(interaction[u]) for u in range(usrnum) if interaction[u]]
    merchant_deg_cnt: dict[int, int] = defaultdict(int)
    for u in range(usrnum):
        if interaction[u]:
            for mid in interaction[u]:
                merchant_deg_cnt[mid] += 1
    merchant_degrees = list(merchant_deg_cnt.values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Yelp-Merchant Dataset Statistics', fontweight='bold')

    axes[0].hist(user_degrees, bins=50, color='#1976D2', alpha=0.8)
    axes[0].set_xlabel('Merchants per user')
    axes[0].set_ylabel('Count')
    axes[0].set_title('User degree distribution')
    axes[0].set_yscale('log')

    axes[1].hist(merchant_degrees, bins=50, color='#F57C00', alpha=0.8)
    axes[1].set_xlabel('Users per merchant')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Merchant degree distribution')
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'dataset_stats.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Saved: dataset_stats.png')
except ImportError:
    print('matplotlib not available, skipping plot')

print(f'Users:          {usrnum:>10,}')
print(f'Merchants:      {merchantnum:>10,}')
print(f'Unique pairs:   {n_pairs:>10,}')
print(f'Total events:   {n_events:>10,}')
print(f'Density:        {density:>10.4f}%')

Saved: dataset_stats.png
Users:             268,653
Merchants:         109,325
Unique pairs:    3,999,575
Total events:    4,190,201
Density:            0.0136%


C:\Users\User\AppData\Local\Temp\ipykernel_8148\3593882562.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Cell 7: Train / test / val split ───────────────────────────────────────────
# Chronological split (seed=SEED):
#   - Last interaction timestamp  → test positive
#   - Second-last on a *different* merchant → val positive
#   - All remaining timestamps    → training
# Only the specific held-out timestamp is removed; earlier visits to the same
# merchant stay in training (correct: multi-visit users should keep history).
# Independent 999-negative samples are drawn for test and val separately.

def negSamp(pos_set: set, samp_size: int, node_num: int) -> list[int]:
    """Sample `samp_size` negative merchant IDs not in `pos_set`."""
    valid = [j for j in range(node_num) if j not in pos_set]
    if not valid:
        valid = list(range(node_num))
    samp_size = min(samp_size, len(valid))
    idx = np.random.choice(len(valid), size=samp_size, replace=False)
    return [valid[i] for i in idx]


def split_test_val(
    interaction: list,
    usrnum: int,
    merchantnum: int,
    test_neg: int = TEST_NEG,
    pick_num: int = PICK_NUM,
    seed: int = SEED,
) -> tuple:
    np.random.seed(seed)

    eligible = [
        u for u in range(usrnum)
        if interaction[u] is not None and len(interaction[u]) >= 3
    ]
    perm     = np.random.permutation(eligible)
    pick_usr = perm[:pick_num]

    tst_int   = [None] * usrnum
    val_int   = [None] * usrnum
    test_rows: list[dict] = []
    val_rows:  list[dict] = []
    skipped = 0

    for u in pick_usr:
        merch = interaction[u]
        if merch is None or len(merch) < 3:
            skipped += 1
            continue

        # Flatten all (timestamp, merchant) events and sort chronologically
        pairs = sorted(
            (ts, mid)
            for mid, tss in merch.items()
            for ts in tss
        )

        tst_time, tst_mid = pairs[-1]

        # Find val: most recent event on a *different* merchant
        val_mid = val_time = None
        for ts, mid in reversed(pairs[:-1]):
            if mid != tst_mid:
                val_mid, val_time = mid, ts
                break
        if val_mid is None:
            skipped += 1
            continue

        # Remove only the held-out timestamp (not all visits to that merchant)
        tss = [t for t in interaction[u][tst_mid] if t != tst_time]
        if tss:
            interaction[u][tst_mid] = tss
        else:
            del interaction[u][tst_mid]

        tss = [t for t in interaction[u].get(val_mid, []) if t != val_time]
        if tss:
            interaction[u][val_mid] = tss
        elif val_mid in interaction[u]:
            del interaction[u][val_mid]

        # Post-deletion check: need at least 1 training interaction
        if sum(1 for tss in interaction[u].values() if tss) < 1:
            skipped += 1
            continue

        tst_int[u] = tst_mid
        val_int[u] = val_mid

        all_pos = set(merch.keys())
        # Independent negative samples for test and val
        test_negs = negSamp(all_pos, test_neg, merchantnum)
        val_negs  = negSamp(all_pos, test_neg, merchantnum)

        test_rows.append({
            'user_id': u + 1, 'merchant_id': tst_mid,
            'time': tst_time, 'neg_merchants': str(test_negs),
        })
        val_rows.append({
            'user_id': u + 1, 'merchant_id': val_mid,
            'time': val_time,  'neg_merchants': str(val_negs),
        })

    pd.DataFrame(test_rows).to_csv(OUT_DIR + 'test_yelp_merchant.csv',  sep='\t', index=False)
    pd.DataFrame(val_rows).to_csv( OUT_DIR + 'val_yelp_merchant.csv',   sep='\t', index=False)

    n_test = sum(1 for x in tst_int if x is not None)
    n_val  = sum(1 for x in val_int  if x is not None)
    print(f'Test users : {n_test:,}')
    print(f'Val  users : {n_val:,}')
    print(f'Skipped    : {skipped:,}')
    return interaction, tst_int, val_int


interaction_train = copy.deepcopy(interaction)
interaction_train, tstInt, valInt = split_test_val(
    interaction_train, usrnum, merchantnum
)

Test users : 10,000
Val  users : 10,000
Skipped    : 0


In [7]:
# ── Cell 8: Build graph matrices ────────────────────────────────────────────────
# global_mat : CSR (usrnum × merchantnum) counting every visit event
# subMat     : list of T binary CSR matrices (first visit per interval per pair)
# timeMat    : CSR storing last time-interval index per (user, merchant)

def trans(
    interaction: list,
    usrnum: int,
    merchantnum: int,
) -> csr_matrix:
    """Global user–merchant CSR; one entry per visit event (multi-visit support)."""
    r, c, d = [], [], []
    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for _ in tss:
                r.append(uid); c.append(mid); d.append(1.0)
    mat = csr_matrix((d, (r, c)), shape=(usrnum, merchantnum))
    print(f'Global matrix: {mat.shape}, {(mat != 0).nnz:,} unique pairs, {len(d):,} total events')
    return mat


def trans_sub(
    interaction: list,
    usrnum: int,
    merchantnum: int,
    graph_num: int,
    minn: int,
    maxx: int,
) -> tuple[list[csr_matrix], csr_matrix]:
    """T binary time-interval CSR matrices + timeMat.

    Each sub-matrix A_t records the first occurrence of a (user, merchant) pair
    within that interval; timeMat records the index of the last interval in
    which the pair appeared.
    """
    interval = (maxx - minn) / graph_num

    rcd: list[tuple[list, list, list]] = [([],[],[]) for _ in range(graph_num)]
    seen: list[set] = [set() for _ in range(graph_num)]
    last_interval: dict[tuple[int,int], int] = {}

    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for ts in tss:
                t   = int((ts - minn) / interval)
                t   = max(0, min(t, graph_num - 1))
                key = (uid, mid)
                if key not in seen[t]:
                    rcd[t][0].append(uid)
                    rcd[t][1].append(mid)
                    rcd[t][2].append(1.0)
                    seen[t].add(key)
                if key not in last_interval or last_interval[key] < t:
                    last_interval[key] = t

    int_mats: list[csr_matrix] = []
    for t in range(graph_num):
        mat = csr_matrix(
            (rcd[t][2], (rcd[t][0], rcd[t][1])),
            shape=(usrnum, merchantnum),
        )
        int_mats.append(mat)
        print(f'  A_{t}: {mat.nnz:,} non-zeros')

    tr, tc, td = [], [], []
    for (u, i), t in last_interval.items():
        tr.append(u); tc.append(i); td.append(t)
    time_mat = csr_matrix((td, (tr, tc)), shape=(usrnum, merchantnum))
    print(f'  timeMat: {time_mat.nnz:,} non-zeros')
    return int_mats, time_mat


global_mat = trans(interaction_train, usrnum, merchantnum)

print(f'\nBuilding T={GRAPH_NUM} time-interval sub-graphs...')
subMat, timeMat = trans_sub(
    interaction_train, usrnum, merchantnum, GRAPH_NUM, minn, maxx
)

Global matrix: (268653, 109325), 3,980,492 unique pairs, 4,170,201 total events

Building T=5 time-interval sub-graphs...
  A_0: 32,561 non-zeros
  A_1: 330,450 non-zeros
  A_2: 914,581 non-zeros
  A_3: 1,540,799 non-zeros
  A_4: 1,213,684 non-zeros
  timeMat: 3,980,492 non-zeros


In [8]:
# ── Cell 9: Build sequence + save all pickles ───────────────────────────────────
# sequence[u] = chronologically-sorted list of merchant IDs from training visits.
# The test-positive merchant is NOT appended (no temporal leak).
# Multi-visit events each contribute one entry, matching global_mat's event count.

sequence: list[list[int]] = []
empty_count = 0
for u in range(usrnum):
    if not interaction_train[u]:
        sequence.append([0])
        empty_count += 1
        continue
    pairs = sorted(
        (ts, mid)
        for mid, tss in interaction_train[u].items()
        for ts in tss
    )
    if not pairs:
        sequence.append([0])
        empty_count += 1
    else:
        sequence.append([mid for _, mid in pairs])

# ── Consistency assertion ────────────────────────────────────────────────────────
seq_events = sum(len(s) for s in sequence) - empty_count
mat_events = int(global_mat.sum())
assert seq_events == mat_events, (
    f'MISMATCH: sequence has {seq_events:,} events but global_mat has {mat_events:,}'
)
print(f'Sequence events : {seq_events:,}')
print(f'Global mat sum  : {mat_events:,}')
print('Sequence-matrix consistency check PASSED.')

# ── Save core pickle files ───────────────────────────────────────────────────────
trn_mat_time = [global_mat, subMat, timeMat]
with open(OUT_DIR + 'trn_mat_time', 'wb') as f: pickle.dump(trn_mat_time, f)
with open(OUT_DIR + 'tst_int',      'wb') as f: pickle.dump(tstInt,       f)
with open(OUT_DIR + 'val_int',      'wb') as f: pickle.dump(valInt,        f)
with open(OUT_DIR + 'sequence',     'wb') as f: pickle.dump(sequence,      f)
print('Saved: trn_mat_time, tst_int, val_int, sequence')

# ── Save test_dict / val_dict (0-indexed uid → neg list) ─────────────────────────
for split, csv_name in [('test', 'test_yelp_merchant.csv'), ('val', 'val_yelp_merchant.csv')]:
    df_split = pd.read_csv(OUT_DIR + csv_name, sep='\t')
    d: dict[int, list[int]] = {}
    for _, row in df_split.iterrows():
        uid  = int(row['user_id']) - 1
        negs = ast.literal_eval(str(row['neg_merchants']))
        d[uid] = negs
    with open(OUT_DIR + f'{split}_dict', 'wb') as f:
        pickle.dump(d, f)
    print(f'Saved: {split}_dict ({len(d):,} users, 0-indexed)')

# ── Save train CSV ───────────────────────────────────────────────────────────────
train_rows: list[dict] = []
for u in range(usrnum):
    if not interaction_train[u]:
        continue
    for mid, tss in interaction_train[u].items():
        for ts in tss:
            train_rows.append({'user_id': u + 1, 'merchant_id': mid, 'time': ts})

df_train = pd.DataFrame(train_rows)
df_train.to_csv(OUT_DIR + 'train_yelp_merchant.csv', sep='\t', index=False)
print(f'Saved: train_yelp_merchant.csv ({len(df_train):,} rows)')

# ── Save edge_weights.pkl for --use_edge_features ────────────────────────────────
# edge_weights[(uid, mid)] = average star rating (float) over all reviews.
# Aligned to the k-core filtered, integer-mapped IDs.
edge_weights: dict[tuple[int,int], float] = {}
for u_str, merchants in filtered.items():
    if u_str not in user2id:
        continue
    uid = user2id[u_str]
    for bid, tss in merchants.items():
        if bid not in merchant2id:
            continue
        mid   = merchant2id[bid]
        stars = raw_stars.get(u_str, {}).get(bid, [])
        edge_weights[(uid, mid)] = float(np.mean(stars)) if stars else 3.0

with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)
print(f'Saved: edge_weights.pkl ({len(edge_weights):,} pairs)')

Sequence events : 4,170,201
Global mat sum  : 4,170,201
Sequence-matrix consistency check PASSED.
Saved: trn_mat_time, tst_int, val_int, sequence
Saved: test_dict (10,000 users, 0-indexed)
Saved: val_dict (10,000 users, 0-indexed)
Saved: train_yelp_merchant.csv (4,170,201 rows)
Saved: edge_weights.pkl (3,999,575 pairs)


In [9]:
# ── Cell 10: Verification & summary ────────────────────────────────────────────

df_test = pd.read_csv(OUT_DIR + 'test_yelp_merchant.csv', sep='\t')
df_val  = pd.read_csv(OUT_DIR + 'val_yelp_merchant.csv',  sep='\t')

train_pairs = set(zip(df_train['user_id'], df_train['merchant_id']))
test_pairs  = set(zip(df_test['user_id'],  df_test['merchant_id']))
val_pairs   = set(zip(df_val['user_id'],   df_val['merchant_id']))

overlap_tr_ts = train_pairs & test_pairs
overlap_tr_vl = train_pairs & val_pairs
overlap_ts_vl = test_pairs  & val_pairs

print(f'Train-Test overlap : {len(overlap_tr_ts):,} pairs')
print(f'Train-Val  overlap : {len(overlap_tr_vl):,} pairs')
print(f'Test-Val   overlap : {len(overlap_ts_vl):,} pairs')
if overlap_tr_ts:
    print('  (Train-Test overlap expected: earlier visits to same merchant remain in train)')
if overlap_tr_vl:
    print('  (Train-Val overlap expected: earlier visits to same merchant remain in train)')

assert len(overlap_ts_vl) == 0, f'FAIL: Test-Val overlap = {len(overlap_ts_vl)} pairs'
print('Test-Val disjointness check PASSED.')

# Sequence leak check
leak_count = sum(
    1 for u in range(usrnum)
    if tstInt[u] is not None and tstInt[u] in sequence[u]
)
if leak_count:
    print(f'Note: {leak_count:,} sequences contain test merchant '
          '(legitimate: user had earlier visits in training period)')
else:
    print('No temporal leak: test merchant not in any training sequence.')

# File inventory
required_files = [
    'trn_mat_time', 'tst_int', 'val_int', 'sequence',
    'test_dict', 'val_dict', 'user2id.pkl', 'merchant2id.pkl',
    'edge_weights.pkl',
    'train_yelp_merchant.csv', 'test_yelp_merchant.csv', 'val_yelp_merchant.csv',
]
print()
print(f'{"File":<30} {"Size"}')
print('-' * 42)
for fname in required_files:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size / 1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<28} {status}')

print()
print('=' * 60)
print('Preprocessing Complete — yelp-merchant')
print('=' * 60)
print(f'Users                : {usrnum:>10,}')
print(f'Merchants            : {merchantnum:>10,}')
print(f'Train events         : {len(df_train):>10,}')
print(f'Unique train pairs   : {(global_mat != 0).nnz:>10,}')
print(f'Test users           : {len(df_test):>10,}')
print(f'Val  users           : {len(df_val):>10,}')
print(f'Negatives per user   : {TEST_NEG:>10,}')
print(f'Time intervals (T)   : {GRAPH_NUM:>10}')
print(f'Output dir           : {OUT_DIR}')

Train-Test overlap : 494 pairs
Train-Val  overlap : 423 pairs
Test-Val   overlap : 0 pairs
  (Train-Test overlap expected: earlier visits to same merchant remain in train)
  (Train-Val overlap expected: earlier visits to same merchant remain in train)
Test-Val disjointness check PASSED.
Note: 494 sequences contain test merchant (legitimate: user had earlier visits in training period)

File                           Size
------------------------------------------
  trn_mat_time                 132342 KB
  tst_int                      290 KB
  val_int                      290 KB
  sequence                     16488 KB
  test_dict                    37144 KB
  val_dict                     37152 KB
  user2id.pkl                  7744 KB
  merchant2id.pkl              3075 KB
  edge_weights.pkl             75380 KB
  train_yelp_merchant.csv      100032 KB
  test_yelp_merchant.csv       68381 KB
  val_yelp_merchant.csv        68382 KB

Preprocessing Complete — yelp-merchant
Users            